# LeetCode 1865 — Finding Pairs With a Certain Sum bằng Python

Link đề: https://leetcode.com/problems/finding-pairs-with-a-certain-sum/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Design

Skeleton LeetCode:

```python
class FindSumPairs:

    def __init__(self, nums1: List[int], nums2: List[int]):
        ...

    def add(self, index: int, val: int) -> None:
        ...

    def count(self, tot: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Thiết kế cấu trúc dữ liệu dựa trên **ràng buộc bất đối xứng**: `nums1` rất nhỏ (`≤ 1000`), `nums2` lớn (`≤ 10⁵`) ⇒ **duyệt mảng nhỏ, tra bảng băm của mảng lớn**.
2. Duy trì một **bảng đếm tần suất** (`Counter`) đồng bộ với mảng khi có cập nhật điểm: bớt giá trị cũ, thêm giá trị mới.
3. Chuyển bài "đếm cặp có tổng bằng `tot`" (Two Sum dạng đếm) thành `Σ cnt2[tot − x]` qua các `x` của `nums1`.
4. Viết **bộ chạy lệnh** (command runner) và đối chiếu ngẫu nhiên với bản vét cạn — kỹ năng bắt buộc khi làm bài Design.
5. Ước lượng chi phí **tổng** qua tất cả lời gọi (`1000` lần `count` × `1000` phần tử) thay vì từng lời gọi.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2`. Cài đặt cấu trúc dữ liệu hỗ trợ hai loại truy vấn:
1. **Cộng** một số nguyên dương vào một phần tử của `nums2` tại chỉ số cho trước.
2. **Đếm** số cặp `(i, j)` sao cho `nums1[i] + nums2[j]` bằng giá trị cho trước (`0 ≤ i < len(nums1)`, `0 ≤ j < len(nums2)`).

Lớp `FindSumPairs`:
- `FindSumPairs(nums1, nums2)` — khởi tạo với hai mảng.
- `add(index, val)` — `nums2[index] += val`.
- `count(tot)` — trả số cặp `(i, j)` với `nums1[i] + nums2[j] == tot`.

**Ví dụ 1:**

```
["FindSumPairs","count","add","count","count","add","add","count"]
[[[1,1,2,2,2,3],[1,4,5,2,5,4]],[7],[3,2],[8],[4],[0,1],[1,1],[7]]
-> [null,8,null,2,1,null,null,11]
```

- `count(7)` = 8: các cặp `2 + 5` (3 × 2 = 6 cặp) và `3 + 4` (1 × 2 = 2 cặp).
- `add(3, 2)` ⇒ `nums2 = [1,4,5,4,5,4]`; `count(8)` = 2 (`3 + 5`); `count(4)` = 1 (`3 + 1`).
- `add(0, 1)`, `add(1, 1)` ⇒ `nums2 = [2,5,5,4,5,4]`; `count(7)` = 11 (`2 + 5`: 3 × 3 = 9, `3 + 4`: 1 × 2 = 2).

**Ràng buộc:** `1 ≤ len(nums1) ≤ 1000`; `1 ≤ len(nums2) ≤ 10⁵`; `1 ≤ nums1[i] ≤ 10⁹`; `1 ≤ nums2[i] ≤ 10⁵`; `0 ≤ index < len(nums2)`; `1 ≤ val ≤ 10⁵`; `1 ≤ tot ≤ 10⁹`; tối đa `1000` lời gọi **mỗi loại** `add` và `count`.

In [1]:
import random
import time
import bisect
from collections import Counter, defaultdict
from typing import List

def run(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["FindSumPairs", "count", "add", "count", "count", "add", "add", "count"]
ex_args = [[[1, 1, 2, 2, 2, 3], [1, 4, 5, 2, 5, 4]], [7], [3, 2], [8], [4], [0, 1], [1, 1], [7]]
ex_out = [None, 8, None, 2, 1, None, None, 11]
for op, a, o in zip(ex_ops, ex_args, ex_out):
    print(f"{op:<13} {str(a):<40} -> {o}")

FindSumPairs  [[1, 1, 2, 2, 2, 3], [1, 4, 5, 2, 5, 4]] -> None
count         [7]                                      -> 8
add           [3, 2]                                   -> None
count         [8]                                      -> 2
count         [4]                                      -> 1
add           [0, 1]                                   -> None
add           [1, 1]                                   -> None
count         [7]                                      -> 11


## 3. Kiến thức nền

### 3.1 `Counter` — bảng đếm tần suất

`Counter(nums2)` ánh xạ **giá trị → số lần xuất hiện**. Tra một khóa không có trả `0` **mà không tạo khóa mới** (khác `defaultdict`).

In [2]:
c = Counter([1, 4, 5, 2, 5, 4])
print(c, "| c[5] =", c[5], "| c[99] =", c[99], "| 99 in c:", 99 in c)

Counter({4: 2, 5: 2, 1: 1, 2: 1}) | c[5] = 2 | c[99] = 0 | 99 in c: False


### 3.2 Đếm cặp có tổng cho trước (Two Sum đếm)

Số cặp `(i, j)` với `a[i] + b[j] = T` bằng `Σᵢ cnt_b[T − a[i]]` — với mỗi phần tử của mảng thứ nhất, số "bạn đời" bên mảng thứ hai là tần suất của `T − a[i]`.

In [3]:
a, b, T = [1, 1, 2, 2, 2, 3], [1, 4, 5, 2, 5, 4], 7
cb = Counter(b)
print("Σ cnt_b[T - x] =", sum(cb[T - x] for x in a), "| vét cạn =", sum(x + y == T for x in a for y in b))

Σ cnt_b[T - x] = 8 | vét cạn = 8


### 3.3 Cập nhật bảng đếm khi một phần tử đổi giá trị

Khi `b[j]` đổi từ `old` sang `new`: `cnt[old] −= 1`, `cnt[new] += 1`, và **ghi lại** `b[j] = new` để lần cập nhật sau biết giá trị cũ. Ba bước, `O(1)`.

### 3.4 Tổng chi phí qua mọi lời gọi (amortized thinking)

Với bài Design, ước lượng: (số lời gọi mỗi loại) × (chi phí mỗi lời gọi) + chi phí khởi tạo. Ở đây: `1000 × O(len(nums1)) = 10⁶` cho `count`, `1000 × O(1)` cho `add`, `O(len(nums2))` để dựng `Counter`.

## 4. Ý tưởng nền tảng / trực giác

`count(tot)` vét cạn là `len(nums1) × len(nums2) = 10⁸` mỗi lần — quá chậm. Nhưng `nums1` chỉ có tối đa `1000` phần tử và **không bao giờ thay đổi**. Vì vậy:
- Giữ `cnt2 = Counter(nums2)` luôn đồng bộ.
- `count(tot) = Σ_{x ∈ nums1} cnt2[tot − x]` — `1000` lần tra bảng.
- `add(index, val)`: chuyển một đơn vị đếm từ giá trị cũ sang giá trị mới.

Tính tay **Ví dụ 1**, lần `count(7)` đầu tiên với `cnt2 = {1:1, 4:2, 5:2, 2:1}`:

| x ∈ nums1 | tot − x | cnt2[tot − x] |
|---|---|---|
| 1 | 6 | 0 |
| 1 | 6 | 0 |
| 2 | 5 | 2 |
| 2 | 5 | 2 |
| 2 | 5 | 2 |
| 3 | 4 | 2 |
| **tổng** | | **8** |

In [4]:
nums1, nums2 = ex_args[0]
cnt2 = Counter(nums2)
rows = [(x, 7 - x, cnt2[7 - x]) for x in nums1]
for r in rows:
    print(r)
print("tổng =", sum(r[2] for r in rows))

(1, 6, 0)
(1, 6, 0)
(2, 5, 2)
(2, 5, 2)
(2, 5, 2)
(3, 4, 2)
tổng = 8


## 5. Cách 1 — Brute Force / baseline: lưu mảng, đếm hai vòng lặp

In [5]:
class FindSumPairsBrute:
    def __init__(self, nums1: List[int], nums2: List[int]):
        self.a = list(nums1)
        self.b = list(nums2)

    def add(self, index: int, val: int) -> None:
        self.b[index] += val

    def count(self, tot: int) -> int:
        return sum(1 for x in self.a for y in self.b if x + y == tot)

out = run(FindSumPairsBrute, ex_ops, ex_args); print(out); assert out == ex_out

[None, 8, None, 2, 1, None, None, 11]


**Độ phức tạp:** `add` `O(1)`; `count` `O(n₁ · n₂)` = `10⁸` mỗi lần, `1000` lần ⇒ `10¹¹` — **TLE** chắc chắn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (bất đối xứng kích thước):** duyệt mảng **nhỏ và cố định** (`nums1`), tra cứu mảng **lớn và thay đổi** (`nums2`) bằng bảng băm.

**Insight 2 (bảng đếm động):** cập nhật điểm trên `nums2` chỉ làm **một** giá trị mất một lần xuất hiện và **một** giá trị được thêm một lần ⇒ `O(1)` để giữ `cnt2` đúng.

**Insight 3 (gom `nums1` theo giá trị, tùy chọn):** `count(tot) = Σ_{v} cnt1[v] · cnt2[tot − v]` qua các **giá trị khác nhau** `v` của `nums1` — nhanh hơn khi `nums1` có nhiều phần tử trùng.

**Invariant:** sau mỗi lời gọi, `cnt2 == Counter(self.nums2)` (bỏ qua khóa có giá trị 0).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter(nums2)` + duyệt `nums1` (bản nộp)

In [6]:
class FindSumPairs:
    def __init__(self, nums1: List[int], nums2: List[int]):
        self.nums1 = nums1
        self.nums2 = nums2
        self.cnt2 = Counter(nums2)

    def add(self, index: int, val: int) -> None:
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.nums2[index] = old + val
        self.cnt2[old + val] += 1

    def count(self, tot: int) -> int:
        cnt2 = self.cnt2
        return sum(cnt2[tot - x] for x in self.nums1)

out = run(FindSumPairs, ex_ops, [[list(ex_args[0][0]), list(ex_args[0][1])]] + ex_args[1:]); print(out); assert out == ex_out

[None, 8, None, 2, 1, None, None, 11]


**Lưu ý:** cài đặt trên **sửa trực tiếp** list `nums2` được truyền vào (giống hầu hết lời giải LeetCode). Khi kiểm thử nhiều lần với cùng dữ liệu, ta truyền **bản sao** để không làm hỏng dữ liệu gốc.

**Độ phức tạp Cách 2:** khởi tạo `O(n₂)`; `add` `O(1)`; `count` `O(n₁)`. Tổng `O(n₂ + 1000 · n₁) ≈ 1.1·10⁶`.

### 7.2 Cách 3 — Gom `nums1` theo giá trị: `Σ cnt1[v] · cnt2[tot − v]`

In [7]:
class FindSumPairsBothCounters:
    def __init__(self, nums1: List[int], nums2: List[int]):
        self.cnt1 = list(Counter(nums1).items())          # (giá trị, số lần) — nums1 không đổi
        self.nums2 = list(nums2)
        self.cnt2 = Counter(nums2)

    def add(self, index: int, val: int) -> None:
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.nums2[index] = old + val
        self.cnt2[old + val] += 1

    def count(self, tot: int) -> int:
        c2 = self.cnt2
        return sum(k * c2[tot - v] for v, k in self.cnt1)

print(run(FindSumPairsBothCounters, ex_ops, ex_args))

[None, 8, None, 2, 1, None, None, 11]


**Độ phức tạp Cách 3:** `count` `O(d₁)` với `d₁` = số giá trị khác nhau của `nums1` (`≤ n₁`).

### 7.3 Cách 4 — Sắp `nums1` + dừng sớm

Vì `nums2[j] ≥ 1` (và chỉ tăng), cặp chỉ có thể tồn tại khi `x < tot`. Sắp các giá trị khác nhau của `nums1` tăng dần và **dừng** khi `x ≥ tot`. Hữu ích khi nhiều truy vấn có `tot` nhỏ so với `nums1`.

In [8]:
class FindSumPairsSortedBreak:
    def __init__(self, nums1: List[int], nums2: List[int]):
        self.vals = sorted(Counter(nums1).items())
        self.nums2 = list(nums2)
        self.cnt2 = Counter(nums2)

    def add(self, index: int, val: int) -> None:
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.nums2[index] = old + val
        self.cnt2[old + val] += 1

    def count(self, tot: int) -> int:
        c2, res = self.cnt2, 0
        for v, k in self.vals:
            if v >= tot:                                    # nums2[j] >= 1 nên không còn cặp
                break
            res += k * c2[tot - v]
        return res

print(run(FindSumPairsSortedBreak, ex_ops, ex_args))

[None, 8, None, 2, 1, None, None, 11]


**Độ phức tạp Cách 4:** khởi tạo thêm `O(n₁ log n₁)`; `count` `O(số giá trị < tot)`.

### 7.4 (Không nên) Cách 5 — Đếm sẵn mọi tổng cặp

Giữ `pair[s]` = số cặp có tổng `s`. `count` là `O(1)`, nhưng khởi tạo `O(n₁ · n₂) = 10⁸` và mỗi `add` phải sửa `n₁` tổng. Chỉ hợp lý khi `count` cực nhiều và `n₂` nhỏ — minh họa trade-off "dồn chi phí vào cập nhật".

In [9]:
class FindSumPairsAllSums:
    def __init__(self, nums1: List[int], nums2: List[int]):
        self.nums1 = list(nums1)
        self.nums2 = list(nums2)
        self.pair = Counter(x + y for x in nums1 for y in nums2)

    def add(self, index: int, val: int) -> None:
        old = self.nums2[index]
        for x in self.nums1:
            self.pair[x + old] -= 1
            self.pair[x + old + val] += 1
        self.nums2[index] = old + val

    def count(self, tot: int) -> int:
        return self.pair[tot]

print(run(FindSumPairsAllSums, ex_ops, ex_args))

[None, 8, None, 2, 1, None, None, 11]


## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**: in `nums2`, các mục khác 0 của `cnt2` và kết quả mỗi lệnh.

In [10]:
def trace(ops, args):
    obj = None
    for op, a in zip(ops, args):
        if obj is None:
            obj = FindSumPairs(list(a[0]), list(a[1])); res = None
        else:
            res = getattr(obj, op)(*a)
        live = {k: v for k, v in sorted(obj.cnt2.items()) if v}
        print(f"{op:<13} {str(a):<38} -> {str(res):<5} nums2={obj.nums2} cnt2={live}")

trace(ex_ops, ex_args)

FindSumPairs  [[1, 1, 2, 2, 2, 3], [1, 4, 5, 2, 5, 4]] -> None  nums2=[1, 4, 5, 2, 5, 4] cnt2={1: 1, 2: 1, 4: 2, 5: 2}
count         [7]                                    -> 8     nums2=[1, 4, 5, 2, 5, 4] cnt2={1: 1, 2: 1, 4: 2, 5: 2}
add           [3, 2]                                 -> None  nums2=[1, 4, 5, 4, 5, 4] cnt2={1: 1, 4: 3, 5: 2}
count         [8]                                    -> 2     nums2=[1, 4, 5, 4, 5, 4] cnt2={1: 1, 4: 3, 5: 2}
count         [4]                                    -> 1     nums2=[1, 4, 5, 4, 5, 4] cnt2={1: 1, 4: 3, 5: 2}
add           [0, 1]                                 -> None  nums2=[2, 4, 5, 4, 5, 4] cnt2={2: 1, 4: 3, 5: 2}
add           [1, 1]                                 -> None  nums2=[2, 5, 5, 4, 5, 4] cnt2={2: 1, 4: 2, 5: 3}
count         [7]                                    -> 11    nums2=[2, 5, 5, 4, 5, 4] cnt2={2: 1, 4: 2, 5: 3}


| lệnh | nums2 sau | cnt2 (khác 0) | kết quả |
|---|---|---|---|
| khởi tạo | `[1,4,5,2,5,4]` | `{1:1, 2:1, 4:2, 5:2}` | — |
| count(7) | | | 8 |
| add(3, 2) | `[1,4,5,4,5,4]` | `{1:1, 4:3, 5:2}` | — |
| count(8) | | | `cnt2[5]` với `x = 3` ⇒ 2 |
| count(4) | | | `cnt2[1]` với `x = 3` ⇒ 1 (`x = 2` cần `cnt2[2] = 0`) |
| add(0, 1) | `[2,4,5,4,5,4]` | `{2:1, 4:3, 5:2}` | — |
| add(1, 1) | `[2,5,5,4,5,4]` | `{2:1, 4:2, 5:3}` | — |
| count(7) | | | `3·cnt2[5] + 1·cnt2[4] = 9 + 2` = 11 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — `Counter(nums2)` + duyệt `nums1`.

In [11]:
class FindSumPairs:

    def __init__(self, nums1: List[int], nums2: List[int]):
        self.nums1 = nums1
        self.nums2 = nums2
        self.cnt2 = Counter(nums2)

    def add(self, index: int, val: int) -> None:
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.nums2[index] = old + val
        self.cnt2[old + val] += 1

    def count(self, tot: int) -> int:
        cnt2 = self.cnt2
        return sum(cnt2[tot - x] for x in self.nums1)

def fresh(args):                       # bản sao input để không sửa dữ liệu gốc
    return [[list(args[0][0]), list(args[0][1])]] + [list(a) for a in args[1:]]

out = run(FindSumPairs, ex_ops, fresh(ex_args)); print(out); assert out == ex_out

[None, 8, None, 2, 1, None, None, 11]


## 10. Vì sao thuật toán đúng

**Invariant:** `cnt2[v]` = số chỉ số `j` với `nums2[j] = v`, với mọi `v`.
- *Khởi tạo:* `Counter(nums2)` đúng theo định nghĩa.
- *`add(index, val)`:* chỉ `nums2[index]` thay đổi, từ `old` thành `old + val`. Đúng một chỉ số rời nhóm `old` và gia nhập nhóm `old + val` (khác `old` vì `val ≥ 1`) ⇒ trừ 1 ở `old`, cộng 1 ở `old + val` giữ invariant. Ta cũng ghi `nums2[index] = old + val` để lần sau đọc đúng `old`.
- *`count(tot)`:* số cặp = `Σᵢ #{j : nums2[j] = tot − nums1[i]} = Σᵢ cnt2[tot − nums1[i]]` (phân hoạch các cặp theo `i`). `count` không sửa trạng thái. ∎

Cách 3 chỉ nhóm các `i` có cùng `nums1[i]`; Cách 4 bỏ các nhóm có `v ≥ tot` vì khi đó `tot − v ≤ 0` mà mọi `nums2[j] ≥ 1` ⇒ đóng góp 0.

### 10.1 Kiểm chứng thực nghiệm invariant

Sau mỗi lệnh trong chuỗi lệnh ngẫu nhiên, so `cnt2` (bỏ khóa bằng 0) với `Counter(nums2)` dựng lại từ đầu.

In [12]:
rng = random.Random(1010)
for _ in range(200):
    n1, n2 = rng.randint(1, 8), rng.randint(1, 10)
    obj = FindSumPairs([rng.randint(1, 6) for _ in range(n1)], [rng.randint(1, 6) for _ in range(n2)])
    for _ in range(30):
        if rng.random() < 0.5:
            obj.add(rng.randrange(n2), rng.randint(1, 4))
        else:
            obj.count(rng.randint(1, 20))
        assert +obj.cnt2 == Counter(obj.nums2)              # +Counter bỏ các khóa <= 0
        assert all(v >= 0 for v in obj.cnt2.values())
print("Invariant cnt2 == Counter(nums2) đúng sau mọi lệnh (200 chuỗi x 30 lệnh)")

Invariant cnt2 == Counter(nums2) đúng sau mọi lệnh (200 chuỗi x 30 lệnh)


## 11. Phân tích Time / Space Complexity

Gọi `n₁ ≤ 1000`, `n₂ ≤ 10⁵`, `Q ≤ 1000` lời gọi mỗi loại.

| Cách | `__init__` | `add` | `count` | Tổng tệ nhất | Space |
|---|---|---|---|---|---|
| 1. Vét cạn | `O(n₂)` | `O(1)` | `O(n₁n₂)` | `10¹¹` | `O(n₁+n₂)` |
| 2. Counter + duyệt nums1 | `O(n₂)` | `O(1)` | `O(n₁)` | `≈ 10⁶` | `O(n₂)` |
| 3. Gom nums1 | `O(n₁+n₂)` | `O(1)` | `O(d₁)` | `≤ 10⁶` | `O(n₂)` |
| 4. Sắp + dừng sớm | `O(n₁ log n₁ + n₂)` | `O(1)` | `O(#v < tot)` | `≤ 10⁶` | `O(n₂)` |
| 5. Mọi tổng | `O(n₁n₂)` | `O(n₁)` | `O(1)` | `10⁸` khởi tạo | `O(n₁n₂)` |

In [13]:
rng = random.Random(1865)
A = [rng.randint(1, 10 ** 5) for _ in range(1000)]
B = [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]
ops = ["FindSumPairs"]; args = [[A, B]]
for _ in range(1000):
    ops.append("add"); args.append([rng.randrange(len(B)), rng.randint(1, 10 ** 5)])
    ops.append("count"); args.append([rng.randint(2, 2 * 10 ** 5)])
for cls in (FindSumPairs, FindSumPairsBothCounters, FindSumPairsSortedBreak):
    t = time.perf_counter(); r = run(cls, ops, fresh(args)); print(f"{cls.__name__:<25}: {(time.perf_counter() - t) * 1000:.0f} ms, tổng count = {sum(x for x in r if x)}")
small_ops, small_args = ["FindSumPairs"], [[A[:200], B[:2000]]]
for _ in range(10):
    small_ops += ["add", "count"]; small_args += [[rng.randrange(2000), rng.randint(1, 10 ** 5)], [rng.randint(2, 2 * 10 ** 5)]]
for cls in (FindSumPairsBrute, FindSumPairsAllSums, FindSumPairs):
    t = time.perf_counter(); r = run(cls, small_ops, fresh(small_args)); print(f"{cls.__name__:<25} (n1=200, n2=2000, 20 lệnh): {(time.perf_counter() - t) * 1000:.0f} ms")

FindSumPairs             : 129 ms, tổng count = 506220


FindSumPairsBothCounters : 149 ms, tổng count = 506220


FindSumPairsSortedBreak  : 102 ms, tổng count = 506220
FindSumPairsBrute         (n1=200, n2=2000, 20 lệnh): 87 ms


FindSumPairsAllSums       (n1=200, n2=2000, 20 lệnh): 69 ms
FindSumPairs              (n1=200, n2=2000, 20 lệnh): 0 ms


## 12. Bộ test

Sinh chuỗi lệnh ngẫu nhiên **hợp lệ** (seed cố định), đối chiếu mọi cài đặt với `FindSumPairsBrute`.

In [14]:
IMPLS = [FindSumPairs, FindSumPairsBothCounters, FindSumPairsSortedBreak, FindSumPairsAllSums]
def check(ops, args):
    ref = run(FindSumPairsBrute, ops, fresh(args))
    for cls in IMPLS:
        assert run(cls, ops, fresh(args)) == ref, (cls.__name__, ops, args)
    return ref

def random_ops(rng, n1, n2, q, vmax=6):
    a = [rng.randint(1, vmax) for _ in range(n1)]; b = [rng.randint(1, vmax) for _ in range(n2)]
    ops, args = ["FindSumPairs"], [[a, b]]
    for _ in range(q):
        if rng.random() < 0.5:
            ops.append("add"); args.append([rng.randrange(n2), rng.randint(1, vmax)])
        else:
            ops.append("count"); args.append([rng.randint(1, 4 * vmax)])
    return ops, args

assert check(ex_ops, ex_args) == ex_out
# chuỗi lệnh tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng vét cạn trong check()
edge = [
    (["FindSumPairs", "count", "count"], [[[1], [1]], [2], [3]], [None, 1, 0]),
    (["FindSumPairs", "add", "count", "add", "count"], [[[1], [1]], [0, 1], [2], [0, 5], [8]], [None, None, 0, None, 1]),
    (["FindSumPairs", "count", "add", "count"], [[[5, 5], [5, 5, 5]], [10], [1, 1], [10]], [None, 6, None, 4]),
    (["FindSumPairs", "count"], [[[10 ** 9], [1]], [10 ** 9 + 1]], [None, 1]),
]
for ops, args, exp in edge:
    assert check(ops, args) == exp, (ops, check(ops, args))
rng = random.Random(18650)
for _ in range(500):
    check(*random_ops(rng, rng.randint(1, 8), rng.randint(1, 10), rng.randint(1, 30)))
for _ in range(5):
    check(*random_ops(rng, 60, 300, 300, vmax=50))
print("Ví dụ, 4 chuỗi tự dựng, 500 chuỗi ngắn, 5 chuỗi dài: OK")

Ví dụ, 4 chuỗi tự dựng, 500 chuỗi ngắn, 5 chuỗi dài: OK


### 12.1 Kiểm tra tính chất

- `Σ_tot count(tot)` qua mọi `tot` khả dĩ = `n₁ · n₂` (mỗi cặp có đúng một tổng).
- `count` **không đổi trạng thái**: gọi hai lần liên tiếp cho cùng kết quả.
- `add` chỉ làm tăng giá trị ⇒ với mọi ngưỡng `M`, số cặp có tổng `> M` không giảm sau `add`.

In [15]:
rng = random.Random(121)
for _ in range(200):
    n1, n2 = rng.randint(1, 6), rng.randint(1, 8)
    a = [rng.randint(1, 5) for _ in range(n1)]; b = [rng.randint(1, 5) for _ in range(n2)]
    obj = FindSumPairs(list(a), list(b))
    for _ in range(10):
        hi = max(a) + max(obj.nums2)
        assert sum(obj.count(t) for t in range(2, hi + 1)) == n1 * n2
        t = rng.randint(2, hi); assert obj.count(t) == obj.count(t)
        M = rng.randint(2, hi)
        before = sum(obj.count(t) for t in range(M + 1, hi + 1))
        obj.add(rng.randrange(n2), rng.randint(1, 3))
        hi2 = max(a) + max(obj.nums2)
        assert sum(obj.count(t) for t in range(M + 1, hi2 + 1)) >= before
print("3 tính chất đúng trên 200 chuỗi lệnh")

3 tính chất đúng trên 200 chuỗi lệnh


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `add` chỉ tăng đếm giá trị mới, quên giảm giá trị cũ

In [16]:
class WrongNoDecrement(FindSumPairs):
    def add(self, index, val):
        self.nums2[index] += val
        self.cnt2[self.nums2[index]] += 1                     # SAI: quên cnt2[old] -= 1

print("sai :", run(WrongNoDecrement, ex_ops, fresh(ex_args)), "\nđúng:", ex_out)

sai : [None, 8, None, 2, 4, None, None, 12] 
đúng: [None, 8, None, 2, 1, None, None, 11]


**Vì sao sai:** giá trị cũ vẫn được đếm như thể phần tử còn đó ⇒ `count` đếm thừa (ở Ví dụ 1, `count(4)` vẫn thấy `nums2[3] = 2` cũ). **Sửa:** `cnt2[old] -= 1` trước khi thêm giá trị mới.

### Lỗi 2 — Cập nhật `cnt2` nhưng quên ghi lại `nums2[index]`

In [17]:
class WrongNoWriteBack(FindSumPairs):
    def add(self, index, val):
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.cnt2[old + val] += 1                             # SAI: nums2[index] vẫn là old

ops = ["FindSumPairs", "add", "add", "count"]; args = [[[1], [1, 9]], [0, 1], [0, 1], [4]]   # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongNoWriteBack, ops, fresh(args)), "| đúng:", run(FindSumPairs, ops, fresh(args)),
      "| brute:", run(FindSumPairsBrute, ops, fresh(args)))

sai : [None, None, None, 0] | đúng: [None, None, None, 1] | brute: [None, None, None, 1]


**Vì sao sai:** lần `add` thứ hai trên cùng chỉ số đọc lại giá trị **cũ** `1` thay vì `2` ⇒ trừ nhầm nhóm, cộng nhầm nhóm. Ví dụ 1 tình cờ không lộ lỗi vì không `add` hai lần cùng chỉ số. **Sửa:** `self.nums2[index] = old + val`.

### Lỗi 3 — Duyệt bảng đếm của `nums2` thay vì `nums1`

In [18]:
class SlowIterateNums2(FindSumPairs):
    def __init__(self, nums1, nums2):
        super().__init__(nums1, nums2); self.cnt1 = Counter(nums1)
    def count(self, tot):
        c1 = self.cnt1
        return sum(k * c1[tot - v] for v, k in self.cnt2.items() if k)   # đúng nhưng O(#giá trị của nums2)

rng = random.Random(13)
ops3 = ["FindSumPairs"] + ["count"] * 200
args3 = [[A, B]] + [[rng.randint(2, 10 ** 5)] for _ in range(200)]
res3 = {}
for cls in (SlowIterateNums2, FindSumPairs):
    t = time.perf_counter(); res3[cls.__name__] = run(cls, ops3, fresh(args3))
    print(f"{cls.__name__:<17}: 200 lần count mất {(time.perf_counter() - t) * 1000:.0f} ms")
assert res3["SlowIterateNums2"] == res3["FindSumPairs"]

SlowIterateNums2 : 200 lần count mất 1236 ms
FindSumPairs     : 200 lần count mất 31 ms


**Vì sao chậm:** kết quả đúng, nhưng `nums2` có tới `10⁵` giá trị khác nhau ⇒ mỗi `count` `10⁵` bước, `1000` lần là `10⁸`. **Sửa:** luôn duyệt mảng **nhỏ** (`nums1`, `≤ 1000`).

### Lỗi 4 — Dùng `defaultdict(int)` và tra `cnt2[tot − x]` làm phình bảng

In [19]:
class LeakyDefaultdict(FindSumPairs):
    def __init__(self, nums1, nums2):
        super().__init__(nums1, nums2)
        self.cnt2 = defaultdict(int, Counter(nums2))

obj_c, obj_d = FindSumPairs(A, list(B)), LeakyDefaultdict(A, list(B))
k0 = len(obj_d.cnt2)
for t in range(10 ** 6, 10 ** 6 + 300):
    assert obj_c.count(t) == obj_d.count(t)
print(f"Counter: {len(obj_c.cnt2)} khóa | defaultdict: {k0} -> {len(obj_d.cnt2)} khóa sau 300 lần count")

Counter: 63198 khóa | defaultdict: 63198 -> 159563 khóa sau 300 lần count


**Vì sao không tốt:** kết quả vẫn đúng, nhưng `defaultdict.__getitem__` **chèn** khóa mới mỗi lần tra trượt ⇒ bộ nhớ tăng tới `Q · n₁ = 10⁶` khóa rác và tra cứu chậm dần. **Sửa:** dùng `Counter` (tra trượt trả 0 không chèn) hoặc `dict.get(k, 0)`.

### Lỗi 5 — Đếm cặp **giá trị** khác nhau thay vì cặp **chỉ số**

In [20]:
class WrongDistinct(FindSumPairs):
    def count(self, tot):
        return sum(1 for x in set(self.nums1) if self.cnt2[tot - x] > 0)   # SAI

print("sai :", run(WrongDistinct, ex_ops, fresh(ex_args)), "\nđúng:", ex_out)

sai : [None, 2, None, 1, 1, None, None, 2] 
đúng: [None, 8, None, 2, 1, None, None, 11]


**Vì sao sai:** đề đếm cặp chỉ số `(i, j)`; giá trị trùng lặp ở cả hai mảng đều được tính nhiều lần (`3 × 2 = 6` cặp `2 + 5`). **Sửa:** nhân tần suất.

### Lỗi 6 — Dựng sẵn mọi tổng cặp (Cách 5) với dữ liệu lớn

In [21]:
for n2 in (500, 1000, 2000):
    t = time.perf_counter(); FindSumPairsAllSums(A, B[:n2])
    print(f"n1=1000, n2={n2:>4}: khởi tạo {(time.perf_counter() - t) * 1000:.0f} ms  (n2 = 10^5 sẽ gấp ~{10 ** 5 // n2} lần)")

n1=1000, n2= 500: khởi tạo 79 ms  (n2 = 10^5 sẽ gấp ~200 lần)


n1=1000, n2=1000: khởi tạo 166 ms  (n2 = 10^5 sẽ gấp ~100 lần)


n1=1000, n2=2000: khởi tạo 340 ms  (n2 = 10^5 sẽ gấp ~50 lần)


**Vì sao không tốt:** `n₁ · n₂ = 10⁸` cặp ⇒ khởi tạo hàng chục giây và bộ nhớ khổng lồ; mỗi `add` còn tốn `O(n₁)`. **Sửa:** chỉ đếm `nums2`, tính tổng khi `count`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài Design có hai mảng **kích thước rất khác nhau** ⇒ duyệt mảng nhỏ, băm mảng lớn.
- "Đếm cặp có tổng `T`" ⇒ `Σ cnt[T − x]` (Two Sum đếm).
- Cập nhật điểm trên mảng có bảng đếm ⇒ "trừ cũ, cộng mới, ghi lại".
- Đọc ràng buộc **số lời gọi** để tính tổng chi phí; đừng tối ưu hóa phần không phải nút thắt.
- Python: `Counter` tra trượt không chèn khóa; `defaultdict` thì có.

## 15. Biến thể của bài

1. **1 Two Sum / 167 / 454 4Sum II**: đếm bộ có tổng bằng `T` qua bảng băm (454: gom hai mảng thành bảng tổng).
2. **Cho phép cập nhật cả `nums1`** — vẫn `O(1)` mỗi `add` nếu giữ `cnt1` (Bài tập 4).
3. **Đếm cặp có tổng trong đoạn `[lo, hi]`** — cần cấu trúc có thứ tự (danh sách sắp + `bisect`, hoặc Fenwick sau nén tọa độ) (Bài tập 5).
4. **Hiệu thay vì tổng**: `nums2[j] − nums1[i] = d` ⇒ `Σ cnt2[x + d]`.
5. **170 Two Sum III — Data structure design** (Premium): `add(number)`, `find(value)` — cùng tinh thần.

### 15.1 Code biến thể: đếm theo hiệu và đối chiếu vét cạn

In [22]:
class FindDiffPairs(FindSumPairs):
    def count_diff(self, d):                       # số cặp nums2[j] - nums1[i] == d
        c2 = self.cnt2
        return sum(c2[x + d] for x in self.nums1)

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 6))]; b = [rng.randint(1, 6) for _ in range(rng.randint(1, 8))]
    obj = FindDiffPairs(list(a), list(b))
    for _ in range(5):
        j, v = rng.randrange(len(b)), rng.randint(1, 3); obj.add(j, v); b[j] += v
        d = rng.randint(-6, 10)
        assert obj.count_diff(d) == sum(y - x == d for x in a for y in b)
print("count_diff khớp vét cạn trên 300 chuỗi")

count_diff khớp vét cạn trên 300 chuỗi


## 16. So sánh các cách làm

| Cách | Ý tưởng | `count` | `add` | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Hai vòng lặp | `O(n₁n₂)` | `O(1)` | Kiểm chứng |
| 2. Counter nums2 | Duyệt nums1, tra cnt2 | `O(n₁)` | `O(1)` | **Bản nộp** |
| 3. Gom nums1 | `Σ cnt1·cnt2` | `O(d₁)` | `O(1)` | nums1 nhiều trùng |
| 4. Sắp + dừng sớm | Bỏ `v ≥ tot` | `O(#v<tot)` | `O(1)` | `tot` thường nhỏ |
| 5. Mọi tổng | Bảng tổng cặp | `O(1)` | `O(n₁)` | Chỉ khi `n₁n₂` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `FindSumPairs([1,2], [1,1,2])`, rồi `count(3)`, `add(2, 1)`, `count(3)`, `count(4)`.

**Bài 2 (tính tay):** Với `nums1 = [2,2,3]`, `nums2 = [1,1,1,2]`: liệt kê `cnt2` sau `add(0, 1)` rồi `add(0, 1)` và tính `count(5)`.

**Bài 3 (điền TODO):** Hoàn thiện `add` của lớp `FindSumPairsTodo` (ba bước).

**Bài 4 (tự viết lại):** Viết lớp `FindSumPairs2` hỗ trợ thêm `add1(index, val)` cộng vào `nums1`, mọi thao tác cập nhật `O(1)`, `count` `O(d₁)`.

**Bài 5 (mini challenge):** Viết lớp `FindSumRange` với `add(index, val)` và `count_range(lo, hi)`: số cặp có `lo ≤ nums1[i] + nums2[j] ≤ hi`. Gợi ý: giữ `nums2` dạng danh sách **đã sắp**; `add` = `bisect` xóa + `insort`; `count_range` = với mỗi `x` hai lần `bisect`. Đối chiếu vét cạn.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class FindSumPairsTodo:
    def __init__(self, nums1, nums2):
        self.nums1, self.nums2, self.cnt2 = nums1, nums2, Counter(nums2)

    def add(self, index, val):
        # TODO: old = self.nums2[index]; giảm cnt2[old]; ghi nums2[index]; tăng cnt2[mới]
        pass

    def count(self, tot):
        return sum(self.cnt2[tot - x] for x in self.nums1)

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class FindSumPairs2:
    def __init__(self, nums1, nums2):
        # TODO: lưu nums1, nums2, cnt1, cnt2
        pass

class FindSumRange:
    def __init__(self, nums1, nums2):
        # TODO: self.sorted2 = sorted(nums2)
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `cnt2 = {1:2, 2:1}`. Mức 2: `count(3)`: `x=1` cần 2 (1 cặp), `x=2` cần 1 (2 cặp) ⇒ 3. Mức 3: sau `add(2,1)`, `cnt2 = {1:2, 3:1}` ⇒ `count(3) = 0 + 2 = 2`, `count(4)`: `x=1` cần 3 (1), `x=2` cần 2 (0) ⇒ 1. Output `[None, 3, None, 2, 1]`.

**Bài 2.** Mức 1: lần 1 `nums2[0]: 1 → 2`. Mức 2: lần 2 `2 → 3` ⇒ `nums2 = [3,1,1,2]`, `cnt2 = {1:2, 2:1, 3:1}`. Mức 3: `count(5)`: `x=2` cần 3 (1 cặp, ×2) = 2; `x=3` cần 2 (1) ⇒ 3.

**Bài 3.** Mức 1: đọc `old` trước. Mức 2: `cnt2[old] -= 1`. Mức 3: ghi `nums2[index] = old + val` và `cnt2[old + val] += 1`.

**Bài 4.** Mức 1: `count` duyệt `cnt1.items()`. Mức 2: `add1` cập nhật `cnt1` giống `add`. Mức 3: bỏ qua khóa có đếm 0 khi duyệt.

**Bài 5.** Mức 1: `bisect_left(sorted2, old)` để xóa đúng một bản. Mức 2: `insort(sorted2, old + val)`. Mức 3: `count_range = Σ_x bisect_right(sorted2, hi − x) − bisect_left(sorted2, lo − x)`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
b1 = run(FindSumPairs, ["FindSumPairs", "count", "add", "count", "count"], [[[1, 2], [1, 1, 2]], [3], [2, 1], [3], [4]])
print("Bài 1:", b1); assert b1 == [None, 3, None, 2, 1]
o = FindSumPairs([2, 2, 3], [1, 1, 1, 2]); o.add(0, 1); o.add(0, 1)
print("Bài 2:", o.nums2, {k: v for k, v in o.cnt2.items() if v}, o.count(5)); assert o.count(5) == 3
class FindSumPairsTodo:
    def __init__(self, nums1, nums2):
        self.nums1, self.nums2, self.cnt2 = nums1, nums2, Counter(nums2)
    def add(self, index, val):
        old = self.nums2[index]
        self.cnt2[old] -= 1
        self.nums2[index] = old + val
        self.cnt2[old + val] += 1
    def count(self, tot):
        return sum(self.cnt2[tot - x] for x in self.nums1)
rng = random.Random(3)
for _ in range(200):
    ops_, args_ = random_ops(rng, rng.randint(1, 5), rng.randint(1, 6), 15)
    assert run(FindSumPairsTodo, ops_, fresh(args_)) == run(FindSumPairsBrute, ops_, fresh(args_))
print("Bài 3: OK")

Bài 1: [None, 3, None, 2, 1]
Bài 2: [3, 1, 1, 2] {1: 2, 2: 1, 3: 1} 3
Bài 3: OK


In [26]:
# Bài 4, 5
class FindSumPairs2:
    def __init__(self, nums1, nums2):
        self.nums1, self.nums2 = list(nums1), list(nums2)
        self.cnt1, self.cnt2 = Counter(nums1), Counter(nums2)
    def _move(self, arr, cnt, index, val):
        old = arr[index]; cnt[old] -= 1; arr[index] = old + val; cnt[old + val] += 1
    def add(self, index, val):
        self._move(self.nums2, self.cnt2, index, val)
    def add1(self, index, val):
        self._move(self.nums1, self.cnt1, index, val)
    def count(self, tot):
        c2 = self.cnt2
        return sum(k * c2[tot - v] for v, k in self.cnt1.items() if k)

class FindSumRange:
    def __init__(self, nums1, nums2):
        self.nums1, self.nums2 = list(nums1), list(nums2)
        self.sorted2 = sorted(nums2)
    def add(self, index, val):
        old = self.nums2[index]
        del self.sorted2[bisect.bisect_left(self.sorted2, old)]
        self.nums2[index] = old + val
        bisect.insort(self.sorted2, old + val)
    def count_range(self, lo, hi):
        s = self.sorted2
        return sum(bisect.bisect_right(s, hi - x) - bisect.bisect_left(s, lo - x) for x in self.nums1)

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 6))]; b = [rng.randint(1, 6) for _ in range(rng.randint(1, 8))]
    o2, orr = FindSumPairs2(a, b), FindSumRange(a, b); a, b = list(a), list(b)
    for _ in range(8):
        if rng.random() < 0.5:
            j, v = rng.randrange(len(b)), rng.randint(1, 3); o2.add(j, v); orr.add(j, v); b[j] += v
        else:
            i, v = rng.randrange(len(a)), rng.randint(1, 3); o2.add1(i, v); a[i] += v
            orr = FindSumRange(a, b)
        t = rng.randint(2, 20); lo = rng.randint(2, 20); hi = lo + rng.randint(0, 6)
        assert o2.count(t) == sum(x + y == t for x in a for y in b)
        assert orr.count_range(lo, hi) == sum(lo <= x + y <= hi for x in a for y in b)
print("Bài 4, 5: OK")

Bài 4, 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class FindSumPairs:
    def __init__(self, nums1, nums2):
        self.nums1, self.nums2 = nums1, nums2
        self.cnt2 = Counter(nums2)
    def add(self, index, val):
        old = self.nums2[index]
        self.cnt2[old] -= 1                 # trừ cũ
        self.nums2[index] = old + val       # ghi lại
        self.cnt2[old + val] += 1           # cộng mới
    def count(self, tot):
        return sum(self.cnt2[tot - x] for x in self.nums1)   # duyệt mảng NHỎ
```

**Checklist:**
- [ ] Duyệt `nums1` (nhỏ), tra `cnt2` (lớn)?
- [ ] `add`: trừ giá trị cũ, **ghi lại** `nums2[index]`, cộng giá trị mới?
- [ ] Đếm cặp **chỉ số** (nhân tần suất), không phải cặp giá trị?
- [ ] Dùng `Counter`/`dict.get` để tra trượt không chèn khóa?
- [ ] Đã đối chiếu chuỗi lệnh ngẫu nhiên với vét cạn?